## PyTorch computer vision

### 0. computer vision libraries in PyTorch

- [`torchvision`](https://docs.pytorch.org/vision/stable/index.html) - base domain library for PyTorch computer vision.

- `torchvision.datasets` - get datasets and data loading functions for computer vision.

- `torchvision.models` - get pretrained computer vision models that we can leverage for our own problems.

- `torchvision.tranforms` - for manipulating our  vision data (images) to be suitable for use with a ML model.

- `torch.utils.data.Dataset` - Base dataset class for PyTorch.

- `torch.utils.data.DataLoader` - Creates a python iterable over a dataset.


In [ ]:
import torch
from torch import nn

import torchvision
from torchvision import datasets
from torchvision import transforms
from torchvision.transforms import ToTensor
from torchvision.io import read_image

import matplotlib.pyplot as plt
import numpy as np

# Check versions
print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")

### 1. Getting a dataset

- The dataset we'll be using is `FashionMNIST` from `torchvision.datasets`.

In [ ]:
# Setup training data
train_data = datasets.FashionMNIST(
    root="data",  # Where to download the data to?
    train=True,  # Do we want the training dataset?
    download=True, # Do we want to download it?
    transform=ToTensor(),  # How do we want to transform the data?
    target_transform=None # How do we want to transforms the labels(targets)?
)


test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
    target_transform=None
)


In [ ]:
len(train_data), len(test_data)

In [ ]:
# See the first training example
image, label = train_data[0]

image, label

In [ ]:
class_names = train_data.classes
class_names

In [ ]:
class_to_idx = train_data.class_to_idx
class_to_idx

### 1.1 Check input and output of our image

In [ ]:
# Check the shape of the image
# It has only 1 color channel becuase the images are black & white
print(f"Image shape: {image.shape} --> [color channels, hieght, width]")
print(f"Image label: {label} --> {class_names[label]}")

### 1.2 Visualizing

In [ ]:
image, label = train_data[0]

print(f"Image shape: {image.shape}")
print(f"Image label: {label}")

In [ ]:
plt.imshow(image.squeeze(),cmap="gray")
plt.title(class_names[label])
plt.axis(False)

In [ ]:
# Check out other images randomly
torch.manual_seed(42)

fig = plt.figure(figsize=(9,9))
rows, cols = 4,4
for i in range(1, rows * cols + 1):
  random_idx = torch.randint(0, len(train_data), size=[1]).item()
  img, label = train_data[random_idx]
  fig.add_subplot(rows, cols, i)
  plt.imshow(img.squeeze(), cmap="gray")
  plt.title(class_names[label])
  plt.axis(False)

Do you think these items of clothing (images) could be modelled with pure linear lines or we'll need `non-linearity`?

In [ ]:
train_data

### 2. Prepare DataLoader

- Right now, our data is in the form of PyTorch Datasets.

- DataLoader turns our dataset into a python iterable.

- More specifically, we want to turn our data into batches (or mini-batches).

Why we need to do so?

1. It is more computationally efficient, as our computing hardware may not be able to look (store in memory) at 60000 images at on hit. So we break it down to 32 images at a time (batch-size = 32).

2. It gives our neural network more chances to update its gradients per epoch.




In [ ]:
from torch.utils.data import DataLoader

# Setup the batch size hyperparameter
BATCH_SIZE = 32

# Turn datasets into iterables (batches)
train_dataloader = DataLoader(dataset=train_data, batch_size=BATCH_SIZE, shuffle=True)

# About the test dataset, the order doesn't matter as the model do not see it during the training phase, so we will
# set "shuffle = False"
test_dataloader = DataLoader(dataset=test_data, batch_size = BATCH_SIZE, shuffle=False)


train_dataloader, test_dataloader




In [ ]:
# Let's check what we have created

print(f"Length of train_dataloader: {len(train_dataloader)} batches of {BATCH_SIZE}")
print(f"Length of test_dataloader: {len(test_dataloader)} batches of {BATCH_SIZE}")

In [ ]:
# Check out what's inside the training dataloader
# iter(train_dataloader) = turns it into an iterator, so you can step through batches one by one
train_features_batch, train_labels_batch = next(iter(train_dataloader))

train_features_batch.shape, train_labels_batch.shape

In [ ]:
# Show a sample
torch.manual_seed(42)

random_idx = torch.randint(0, len(train_features_batch), size=[1]).item()
img, label = train_features_batch[random_idx], train_labels_batch[random_idx]

plt.imshow(img.squeeze(), cmap="gray")
plt.title(class_names[label])
plt.axis(False)
print(f"Image size: {img.shape}")
print(f"Label: {label}, {class_names[label]}")


### 3. Model 0: Build a baseline model

when starting to build a series of machine learning modelling experiences, the base practice is to start with the baseline model.

A **baseline model** is a model we will try and improve upon with subsequent models/experiences.

In other words:

- Start simply and add complexity when necessary.



In [ ]:
# Create a flatten layer
flatten_model = nn.Flatten()

# Get a single sample
x = train_features_batch[0]

# Flatten the sample
out = flatten_model(x)

# Print out what happend:
print(f"Shape before flattening: {x.shape} -> [color_channels, height, width]")
print(f"Shape after flattening: {out.shape} -> [color_channels, height*width]")

In [ ]:
class FashionMNISTModel_v0(nn.Module):
  def __init__(self, input_shape: int, hidden_units: int, output_shape: int):
    super().__init__()
    self.layer_stack = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=input_shape, out_features=hidden_units),
        nn.Linear(in_features=hidden_units, out_features=output_shape)
    )


  def forward(self, x):
    return self.layer_stack(x)


In [ ]:
torch.manual_seed(42)

# Setup model with input parameters

model_0 = FashionMNISTModel_v0(
    input_shape=784,  # This is 28*28
    hidden_units=10,  # how many units in the hidden layer
    output_shape=len(class_names)  # one for every class
    ).to('cpu')

model_0

In [ ]:
# Test
dummy_x = torch.rand([1,1, 28, 28])
model_0(dummy_x)

In [ ]:
model_0.state_dict()

### 3.1 Setup loss, optimizer and evaluation metrics

* Loss function - since we're working with multi-class data, our loss function will be `nn.CrossEntropyLoss()`

* Optimizer - our optimizer `torch.optim.SGD()` (stochastic gradient descent)

* Evaluation metric - Since we're working on a classification problem, let's use accuracy as our evaluation metric

In [ ]:
from urllib import request
import requests
from pathlib import Path

# Download helper function from Learn PyTorch repo
if Path("helper_functions.py").is_file():
  print("helper_functions.py already exists, skipping download")
else:
  print("Downloading helper_functions.py")
  request = requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/helper_functions.py")
  with open("helper_functions.py", "wb") as f:
    f.write(request.content)


In [ ]:
# Import accuracy metric

from helper_functions import accuracy_fn

# Setup loss and optimizer

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params=model_0.parameters(), lr=0.1)



### 3.2. Creating a function to time our experiments

Machine learning is very experimental.

Two of main things we'll often want to track are:

1. Model performance (loss, accuracy values and etc)

2. How fast it runs

In [ ]:
from timeit import default_timer as timer

def print_train_time(start: float, end: float, device: torch.device = None):
  """Prints difference between start and end time."""
  total_time = end - start
  print(f"Train time on {device}: {total_time:.3f} seconds")
  return total_time

In [ ]:
start_time = timer()
# Some code...
end_time = timer()
print_train_time(start=start_time, end=end_time, device='cpu')

### 3.3. Creating a training loop and training a model on batches of data

1. Loop through epochs.

2. Loop through training batches, perform training steps, calculate the train loss **per batch**.


3. Loop through testing batches, perform testing steps, calculate the test loss **per batch**.

4. Print what is happening.

5. Time it all (for us).


In [ ]:
from tqdm.auto import tqdm

'''
Training:

  1- predict

  2- measure error

  3- compute gradients

  4- update weights

'''

# Set the seed and start the timer
torch.manual_seed(42)
train_time_start_on_cpu = timer()

# Set the number of epochs (we'll keep this small for faster training time)
epochs = 3

# Create a training and testing loop
for epoch in tqdm(range(epochs)):
  print(f"Epoch: {epoch}\n-------")
  ### Training
  train_loss = 0
  # Add a loop to loop through the training batches
  for batch, (X, y) in enumerate(train_dataloader):
    model_0.train()
    # 1. Forward pass
    y_pred = model_0(X)

    # 2. Calculate the loss (per batch)
    loss = loss_fn(y_pred, y)
    train_loss += loss  # accumulate train loss


    # 3. Optimizer zero grad
    optimizer.zero_grad()

    # 4. Loss backward
    loss.backward()

    # 5. optimizer step
    optimizer.step()

    # Print out what is happening
    if batch % 400 == 0:
      print(f"Looked at {batch * len(X)}/{len(train_dataloader.dataset)} samples.")

# Divide total train loss by length of train dataloader (average loss)
train_loss /= len(train_dataloader)

### Testing
test_loss, test_acc = 0, 0
model_0.eval()
with torch.inference_mode():
  for X_test, y_test in test_dataloader:
    # 1. Forward pass
    test_pred = model_0(X_test)

    # 2. Calculate loss (accumulatively)
    test_loss += loss_fn(test_pred, y_test)

    # 3. Calculate accuracy
    test_acc += accuracy_fn(y_true=y_test, y_pred=test_pred.argmax(dim=1))

  # Calculate the test loss average per batch
  test_loss /= len(test_dataloader)

  # Calculate the test accuracy average per batch
  test_acc /= len(test_dataloader)

# Print out what is happening
print(f"\nTrain loss: {train_loss:.4f} | Test loss: {test_loss:.4f}, Test acc: {test_acc:.4f}")


# Calculate training time
train_time_end_on_cpu = timer()
total_train_time_model_0 = print_train_time(start=train_time_start_on_cpu, end=train_time_end_on_cpu, device=str(next(model_0.parameters()).device))

### 4. Make predictions and get Model 0 results


In [ ]:
torch.manual_seed(42)
def eval_model(model: torch.nn.Module,
               data_loader: torch.utils.data.dataloader,
               loss_fn: torch.nn.Module,
               accuracy_fn):
  """Returns a dictionary containing the results of model predicting on data_loader."""
  loss, acc = 0,0
  model.eval()
  with torch.inference_mode():
    for X, y in tqdm(data_loader):
      # Make predictions
      y_pred = model(X)

      # Accumulate the loss and acc values per batch
      loss += loss_fn(y_pred, y)
      acc += accuracy_fn(y_true=y, y_pred= y_pred.argmax(dim=1))

    # Scale the loss and acc to average the loss/acc per batch
    loss /= len(data_loader)
    acc /= len(data_loader)

  return {"model_name": model.__class__.__name__, # this only works when model was created with a class
          "model_loss": loss.item(),
          "model_acc": acc}


# Calculate model_0 results on test dataset
model_0_results = eval_model(model=model_0,
                             data_loader=test_dataloader,
                             loss_fn=loss_fn,
                             accuracy_fn=accuracy_fn)

model_0_results

### 5. Setup device agnostic code


In [ ]:
torch.cuda.is_available()

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

### 6. Building a better model using `non-linearity`


In [ ]:
# Create a model with non-linear and linear layers
class FashionMNISTModel_v1(nn.Module):
  def __init__(self, input_shape: int, hidden_units: int, output_shape: int):
    super().__init__()
    self.layer_stack = nn.Sequential(
        nn.Flatten(), # Flatten input into a single vector
        nn.Linear(in_features=input_shape, out_features=hidden_units),
        nn.ReLU(),
        nn.Linear(in_features=hidden_units, out_features=output_shape),
        nn.ReLU()
    )


  def forward(self, x: torch.Tensor):
    return self.layer_stack(x)



In [ ]:
# Create an instance of model_1
torch.manual_seed(42)

# Setup model with input parameters

model_1 = FashionMNISTModel_v1(
    input_shape=784,  # This is 28*28
    hidden_units=10,  # how many units in the hidden layer
    output_shape=len(class_names)  # one for every class
    ).to(device)

model_0

### 6.1. Setup loss, optimizer and evaluation metrics

In [ ]:
from helper_functions import accuracy_fn

# Setup loss and optimizer

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params=model_1.parameters(), lr=0.1)


### 6.2. Training and testing loop

Let's create a function for:

- Training loop: `train_step()`

- Testing loop: `test_step()`

In [ ]:
def train_step(model: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               accuracy_fn,
               device: torch.device = device):
  train_loss, train_acc = 0,0

  # Put model on training mode
  model.train()
  for batch, (X, y) in enumerate(data_loader):
    # Put data on target device
    X, y = X.to(device), y.to(device)

    # 1. Forward pass
    y_pred = model(X)

    # 2. Calculate the loss/acc (per batch)
    loss = loss_fn(y_pred, y)
    train_loss += loss.item()  # accumulate train loss
    train_acc += accuracy_fn(y_true=y, y_pred=y_pred.argmax(dim=1)) # From logits to prediction labels

    # 3. Optimizer zero grad
    optimizer.zero_grad()

    # 4. Loss backward
    loss.backward()

    # 5. Optimizer step
    optimizer.step()

  # Divide total train loss/acc by length of train dataloader (average loss/acc per batch)
  train_loss /= len(data_loader)
  train_acc /= len(data_loader)
  return train_loss, train_acc

In [ ]:
def test_step(model: torch.nn.Module,
              data_loader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module,
              accuracy_fn,
              device: torch.device = device):
  test_loss, test_acc = 0,0
  # Put model on eval mode
  model.eval()

  # Turn on inference mode context manager
  with torch.inference_mode():
    for X, y in data_loader:
      # Put data on target device
      X, y = X.to(device), y.to(device)

      # 1. Forward pass
      test_pred = model(X)

      # 2. Calculate the loss/acc (per batch)
      test_loss += loss_fn(test_pred, y).item()
      test_acc += accuracy_fn(y_true=y, y_pred=test_pred.argmax(dim=1)) # go from logits to prediction labels

  # Adjust metrics and print out
  test_loss /= len(data_loader)
  test_acc /= len(data_loader)
  return test_loss, test_acc

In [ ]:
torch.manual_seed(42)

# Measure time
from timeit import default_timer as timer
train_time_start_on_gpu = timer()

epochs = 3

for epoch in tqdm(range(epochs)):
    print(f"Epoch: {epoch}\n-------")
    train_loss, train_acc = train_step(model=model_1,
                                       data_loader=train_dataloader,
                                       loss_fn=loss_fn,
                                       optimizer=optimizer,
                                       accuracy_fn=accuracy_fn,
                                       device=device)

    test_loss, test_acc = test_step(model=model_1,
                                    data_loader=test_dataloader,
                                    loss_fn=loss_fn,
                                    accuracy_fn=accuracy_fn,
                                    device=device)
    print(f"Train loss: {train_loss:.4f} | Train acc: {train_acc:.4f} | Test loss: {test_loss:.4f} | Test acc: {test_acc:.4f}")

train_time_end_on_gpu = timer()
total_train_time_model_1 = print_train_time(start=train_time_start_on_gpu,
                                            end=train_time_end_on_gpu,
                                            device=device)

In [ ]:
torch.manual_seed(42)
def eval_model(model: torch.nn.Module,
               data_loader: torch.utils.data.dataloader,
               loss_fn: torch.nn.Module,
               accuracy_fn,
               device: torch.device=device):
  """Returns a dictionary containing the results of model predicting on data_loader."""
  loss, acc = 0,0
  model.eval()
  with torch.inference_mode():
    for X, y in tqdm(data_loader):
      # target device
      X,y = X.to(device), y.to(device)
      # Make predictions
      y_pred = model(X)

      # Accumulate the loss and acc values per batch
      loss += loss_fn(y_pred, y)
      acc += accuracy_fn(y_true=y, y_pred= y_pred.argmax(dim=1))

    # Scale the loss and acc to average the loss/acc per batch
    loss /= len(data_loader)
    acc /= len(data_loader)

  return {"model_name": model.__class__.__name__, # this only works when model was created with a class
          "model_loss": loss.item(),
          "model_acc": acc}


In [ ]:
# Get model_1 results dictionary
model_1_results = eval_model(model=model_1,
                             data_loader=test_dataloader,
                             loss_fn=loss_fn,
                             accuracy_fn=accuracy_fn,
                             device=device)

model_1_results

### Model_2: Building a Convolutional Neural Network (CNN)

- CNN's are also known as ConvNets.

- CNN's are known for their capabilities to find patterns in visual data.

- To find out what's happening inside the CNN, check out this website: https://poloclub.github.io/cnn-explainer/#article-convolution

In [ ]:
# Create a convolutional nueral network

# Kernel size: the size of the filter window that scans the image, such as 3×3 or 5×5.
# Stride: how many pixels the filter moves each step across the input, such as 1 or 2.
# Padding: extra pixels (usually zeros) added around the input border so the filter can cover the edges and control output size.

class FashionMNISTModel_v2(nn.Module):
  """
  model architecture that replicate the TinyVGG (from CNN explane website)
  """
  def __init__(self, input_shape: int, hidden_units: int, output_shape: int):
    super().__init__()
    self.conv_block_1 = nn.Sequential(
        nn.Conv2d(in_channels=input_shape, out_channels=hidden_units, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2)
    )

    self.conv_block_2 = nn.Sequential(
        nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2)
    )
    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=hidden_units*7*7, out_features=output_shape)
    )

  def forward(self, x: torch.Tensor):
    x = self.conv_block_1(x)
    # print(x.shape)
    x = self.conv_block_2(x)
    # print(x.shape)
    x = self.classifier(x)
    return x

In [ ]:
image.shape

In [ ]:
# Instantiate the model
torch.manual_seed(42)
# input shape = # color channels of the image
model_2 = FashionMNISTModel_v2(input_shape=1, hidden_units=10, output_shape=len(class_names)).to(device)
model_2

### 7.1. Stepping through `nn.conv2d`

- Documentation: https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html

In [ ]:
torch.manual_seed(42)

# Create a batch of images
images = torch.randn(size=(32, 3, 64, 64)).to(device)

# Create a singular image
test_image = images[0]

print(f"Image batch shape: {images.shape} -> [batch_size, color_channels, height, width]")
print(f"Single image shape: {test_image.shape} -> [color_channels, height, width]")
print(f"test image\n {test_image}")

In [ ]:
# Create a single conv2d layer
conv_layer = nn.Conv2d(in_channels=3, out_channels=10, kernel_size=(3,3), stride=1, padding=0).to(device)

# Pass the data through the convolutional layer
# Point: As the conv_layer expect 4-dim input, we need to add an extra dim to the test_image --> However, in new version of pytorch, it will done automatically

# test_image = test_image.unsqueeze(dim=0)

conv_out = conv_layer(test_image)
conv_out

As we can see, the shape of the output layer changed compare to the intput which was the test_image   

In [ ]:
test_image.shape, conv_out.shape

### 7.2. Stepping through `nn.MaxPool2d()`

Documentation: https://docs.pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html

In [ ]:
# Print out original shape without unsqueezed dimension

print(f"Test image original shape: {test_image.shape}")
print(f"Test image with unsqueezed dimension: {test_image.unsqueeze(dim=0).shape}")

# Create a sample nn.MaxPool2d layer
max_pool_layer = nn.MaxPool2d(kernel_size=2) # equal to (2,2)

# Pass data through just conv_layer
test_image_through_conv = conv_layer(test_image.unsqueeze(dim=0))
print(f"Test image through conv_layer: {test_image_through_conv.shape}")

# Then pass data through the max pool layer
test_image_through_pool = max_pool_layer(test_image_through_conv)
print(f"Test image through max pool layer: {test_image_through_pool.shape}")

In [ ]:
# For better understanding, let's create a random tensor with a similar number of dimensions to our shape
torch.manual_seed(42)

random_tensor = torch.randn(size=(1, 1, 2, 2))
print(f"\nRandom tensor: {random_tensor}")
print(f"Random tensor shape: {random_tensor.shape}")

# Create a max pool layer
max_pool_layer = nn.MaxPool2d(kernel_size=2)

# Pass the random tensor through it
max_pool_tensor = max_pool_layer(random_tensor)
print(f"\nMax pool tensor: {max_pool_tensor}")
print(f"Max pool tensor shape: {max_pool_tensor.shape}")

In [ ]:
plt.imshow(image.squeeze(dim=0), cmap="gray")

In [ ]:
# Pass the above image to the CNN model
model_2(image.unsqueeze(0).to(device))

### 7.3. Setup a loss and optimizer function for `model_2`

In [ ]:
from helper_functions import accuracy_fn

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params=model_2.parameters(), lr=0.1)

### 7.4. Create the training and testing loops for `model_2`

In [ ]:
import torch
from tqdm.auto import tqdm
from timeit import default_timer as timer

# Using pre-defined training and testing functions
torch.manual_seed(42)
torch.cuda.manual_seed(42)

# Measure time
train_time_start_model_2 = timer()

# Train and test the model
epochs = 3
for epoch in tqdm(range(epochs)):
    print(f"Epoch: {epoch}\n------")
    train_loss, train_acc = train_step(model=model_2,
               data_loader=train_dataloader,
               loss_fn=loss_fn,
               optimizer = optimizer,
               accuracy_fn=accuracy_fn,
               device=device)

    test_loss, test_acc = test_step(model=model_2,
              data_loader=test_dataloader,
              loss_fn=loss_fn,
              accuracy_fn=accuracy_fn,
              device=device)
    print(f"Train loss: {train_loss:.4f} | Train acc: {train_acc:.4f} | Test loss: {test_loss:.4f} | Test acc: {test_acc:.4f}")

train_time_end_model_2 = timer()

total_train_time_model_2 = print_train_time(start=train_time_start_model_2, end=train_time_end_model_2, device=device)

In [ ]:
# Evaluating
model_2_results = eval_model(model=model_2,
                             data_loader=test_dataloader,
                             loss_fn=loss_fn,
                             accuracy_fn=accuracy_fn,
                             device=device)
model_2_results

### Compare model results and training time


In [ ]:
import pandas as pd
compare_results = pd.DataFrame([model_0_results,
                               model_1_results,
                               model_2_results])
compare_results

In [ ]:
# Add training time to results comparison
compare_results["training time"] = [total_train_time_model_0,
                                    total_train_time_model_1,
                                    total_train_time_model_2]

compare_results

In [ ]:
# Visualize our model results
compare_results.set_index("model_name")["model_acc"].plot(kind="barh")
plt.xlabel("Model Accuracy")
plt.ylabel("Model Name")

### 9. Make and evaluate random predictions with best model

In [ ]:
def make_predictions(model: torch.nn.Module,
                     data: list,
                     device: torch.device = device):

  pred_probs = []
  model.to(device)
  model.eval()
  with torch.inference_mode():
    for sample in data:
      # Prepare the sample
      sample = torch.unsqueeze(sample, dim=0).to(device)
      # Forward pass
      pred_logits = model(sample)
      # Get prediction probability
      pred_prob = torch.softmax(pred_logits.squeeze(), dim=0)
      # Get pred_prob off the GPU
      pred_probs.append(pred_prob.cpu())
  # stack the pred_probs to turn list into the tensor
  return torch.stack(pred_probs)


In [ ]:
import random
# random.seed(42)
test_samples = []
test_labels = []

for sample, label in random.sample(list(test_data), k=9):
  test_samples.append(sample)
  test_labels.append(label)


# View the first sample shape
test_samples[0].shape

In [ ]:
plt.imshow(test_samples[0].squeeze(), cmap="gray")
plt.title(class_names[test_labels[0]])

In [ ]:
# Make predictions
pred_probs = make_predictions(model=model_2, data=test_samples)

# View the first 2 predictions probabilities
pred_probs[:2]

In [ ]:
# Convert predition probabilities to labels
pred_classes = pred_probs.argmax(dim=1)
pred_classes

In [ ]:
# Plot predictions
plt.figure(figsize=(9,9))
nrows = 3
ncols = 3
for i, sample in enumerate(test_samples):
  # Create subplot
  plt.subplot(nrows, ncols, i+1)
  # Plot the target image
  plt.imshow(sample.squeeze(), cmap="gray")
  # Find the prediction label
  pred_label = class_names[pred_classes[i]]
  # Get the truth label
  truth_label = class_names[test_labels[i]]

  # Create a title for the plot
  title_text = f"Pred: {pred_label} | Truth: {truth_label}"

  # Check for equality between pred and truth and change color of title text
  if pred_label == truth_label:
    plt.title(title_text, fontsize=10, c="g")
  else:
    plt.title(title_text, fontsize=10, c="r")

  plt.axis(False)

### 10. Making a confusion matrix for futher prediction evaluation

A confusion matrix is a fantastic way to evaluate our classification model visually.

1. Make predictions with our trained model on the test dataset.

2. Make a confusion matrix using `torchmetrics.ConfusionMatrix`

3. Plot the confusion matrix using `mlxtend.plotting.plot_confusion_matrix()`

In [ ]:
# Import tqdm.auto
from tqdm.auto import tqdm

# 1. Make prediction
y_preds = []
model_2.to(device)
model_2.eval()
with torch.inference_mode():
  for X, y in tqdm(test_dataloader, desc="Making predictions"):
    X, y = X.to(device), y.to(device)
    # Do the forward pass
    y_logits = model_2(X)
    # Turn predictions from logits --> prediction probabilities --> prediction labels
    y_pred = torch.softmax(y_logits.squeeze(), dim=0).argmax(dim=1)
    # Put prediction on cpu
    y_preds.append(y_pred.cpu())

# Concatenate list of predictions into a tensor
# print(y_preds)
y_pred_tensor = torch.cat(y_preds)
y_pred_tensor, len(y_pred_tensor)


In [ ]:
# Installing the required packages
try:
  import torchmetrics, mlxtend
  print("torchmetrics and mlxtend are already installed")
except:
  !pip install -q torchmetrics
  import torchmetrics


In [ ]:
import torchmetrics
torchmetrics.__version__

In [ ]:
from torchmetrics import ConfusionMatrix
from mlxtend.plotting import plot_confusion_matrix

# Setup confusion instance and compare predictions to taegets
confmat = ConfusionMatrix(task="multiclass", num_classes=len(class_names))
confmat_tensor = confmat(preds=y_pred_tensor, target=test_data.targets)
confmat_tensor

In [ ]:
# Plot the confusion matrix
fig, ax = plot_confusion_matrix(conf_mat=confmat_tensor.numpy(),
                                  class_names=class_names,
                                  figsize=(10,10))

### 11. Save and load best performing model

In [ ]:
from pathlib import Path

# Create model directory path
MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)

# Create model save path
MODEL_NAME = "03_pytorch_computer_vision_model.pth"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME

# Save the model state dict
print(f"Saving model to: {MODEL_SAVE_PATH}")
torch.save(obj=model_2.state_dict(), f=MODEL_SAVE_PATH)


In [ ]:
# Create a new instance
torch.manual_seed(42)
loaded_model_2 = FashionMNISTModel_v2(input_shape=1, hidden_units=10, output_shape=len(class_names))

# Load in the save state_dict()
loaded_model_2.load_state_dict(torch.load(f=MODEL_SAVE_PATH))

# Send the model to target device
loaded_model_2.to(device)

In [ ]:

model_2_results

In [ ]:
# Evaluate the loaded model
torch.manual_seed(42)
loaded_model_2_results = eval_model(model=loaded_model_2,
                                    data_loader=test_dataloader,
                                    loss_fn=loss_fn,
                                    accuracy_fn=accuracy_fn,
                                    device=device)

loaded_model_2_results

In [ ]:
# Check if the model results are close together
torch.isclose(torch.tensor(model_2_results["model_loss"]),
              torch.tensor(loaded_model_2_results["model_loss"]),
              atol=1e-02)